# Invariant EKF Covariance Calibration

Notebook for the trimmed covariance-calibration run. Core replay, training, and evaluation utilities are imported from the package; this notebook configures paths, runs checks, and records results.

Environment setup: CUDA float64, project paths, output folders, and imports.


In [1]:
# 01 environment: CUDA-first, float64, bundle paths
import os
import sys
import time
from dataclasses import replace
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch

def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "estimation_calibration_cuda" / "invariant_ekf.py").exists():
            return p
    raise FileNotFoundError("project root not found")

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))
DATA_ROOT = Path(os.environ.get("INEKF_DATA_ROOT", ROOT / "data" / "datasets_v0"))
OUT = ROOT / "runs" / "covariance_calibration"
PLOTS = OUT / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

from estimation_calibration_cuda.covariance_calibration import (
    GROUP_ORDER, GROUP_COLOR, CalibrationConfig, make_device, assert_cuda_float64,
)
from estimation_calibration_cuda.invariant_ekf import (
    replay_inekf_torch, start_filter, run_rows, detach_filter, precompute_contact_changes
)

CONFIG = CalibrationConfig()
torch.set_default_dtype(CONFIG.dtype)
device = make_device(require_cuda=True)  # this notebook is CUDA-first
dtype = CONFIG.dtype
torch.manual_seed(0)
DD = dict(dtype=dtype, device=device)
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

NVIDIA GeForce RTX 5090 Laptop GPU | torch 2.12.0+cu130


Replay checks: parity against golden references and chunked-continuation behavior.


In [2]:
# 02 replay checks: synthetic sequence, G1 slice, and chunked continuation.
gs = np.load(OUT / "gate_c_golden_synthetic.npz")
gg = np.load(OUT / "gate_c_golden_g1_slice.npz")

def covs_from(z):
    return {k: torch.as_tensor(z[k], **DD) for k in ["Qg", "Qa", "Qbg", "Qba", "Qc"]}

def tensors_from(z):
    return (torch.as_tensor(z["X0"], **DD), torch.as_tensor(z["theta0"], **DD),
            torch.as_tensor(z["P0"], **DD), torch.as_tensor(z["imu"] if "imu" in z
            else z["imu_shifted"], **DD), torch.as_tensor(z["p_meas"], **DD),
            torch.as_tensor(z["R_kin"], **DD))

for name, z, tol in [("synthetic", gs, 1e-10), ("g1_slice", gg, 1e-9)]:
    X0, th0, P0, imu, p, Rk = tensors_from(z)
    assert_cuda_float64(X0, th0, P0, imu, p, Rk)
    out = replay_inekf_torch(X0, th0, P0, covs_from(z), imu, float(z["dt"]),
                                     p, z["flags"], Rk)
    d = max(float((out["R_WB"] - torch.as_tensor(z["R_np"], **DD)).abs().max()),
            float((out["v_W"] - torch.as_tensor(z["v_np"], **DD)).abs().max()),
            float((out["p_W"] - torch.as_tensor(z["p_np"], **DD)).abs().max()),
            float((out["final_P"] - torch.as_tensor(z["final_P"], **DD)).abs().max()))
    print(f"Replay check [{name}]: max abs diff = {d:.2e}")
    assert d < tol, (name, d)
    if name == "synthetic":
        got = out["final_estimated_contact_positions"]
        want = dict(zip(z["final_contact_ids"].tolist(), z["final_contact_cols"].tolist()))
        assert got == want, (got, want)  # dynamic dims preserved after add/remove

# chunked continuation == monolithic (bitwise), with and without precomputed events
X0, th0, P0, imu, p, Rk = tensors_from(gg)
covs = covs_from(gg)
out_m = replay_inekf_torch(X0, th0, P0, covs, imu, float(gg["dt"]), p,
                                   gg["flags"], Rk)
gg_changes = precompute_contact_changes(gg["flags"])
for use_pre in (False, True):
    filt = start_filter(X0, th0, P0, covs, gg["flags"][0], p[0], Rk)
    vs = [filt.X[0:3, 3][None]]
    a = 1
    while a < p.shape[0]:
        b = min(a + 100, p.shape[0])
        kw = dict(changes_list=gg_changes[a:b]) if use_pre else {}
        oc = run_rows(filt, imu[a - 1:b - 1], float(gg["dt"]), p[a:b], gg["flags"][a:b],
                      gg["flags"][a - 1], Rk, **kw)
        vs.append(oc["v_W"])
        detach_filter(filt)
        a = b
    v_chunked = torch.cat(vs)
    assert torch.equal(v_chunked, out_m["v_W"]), f"chunked (pre={use_pre}) must be exact"
print("Gate C: PASS (parity, dynamic dims, chunked continuation exact, "
      "precomputed contact events exact)")


Replay check [synthetic]: max abs diff = 2.25e-14


Replay check [g1_slice]: max abs diff = 3.12e-14


Replay checks: PASS (dynamic dims, chunked continuation, precomputed contact events)


Full-SPD Cholesky covariance modules, one per process and measurement covariance group.


In [3]:
# 03 full-SPD Cholesky covariance modules
from estimation_calibration_cuda.covariance_calibration import (
    INIT_STD, FLOOR, COV_KEY, SPD3, CovarianceModel, make_cov_modules, build_covs,
)

# unit checks
_m = make_cov_modules(device=device)
for name in GROUP_ORDER:
    C = _m[name].cov()
    assert_cuda_float64(C)
    # softplus floor offset shifts the diagonal by ~2*std*sqrt(floor): allow 1%
    assert float((C - INIT_STD[name] ** 2 * torch.eye(3, **DD)).abs().max()) < 1e-2 * INIT_STD[name] ** 2
    eig = torch.linalg.eigvalsh(C)
    assert float(eig.min()) > FLOOR[name] / 2
    C.sum().backward()
    assert torch.isfinite(_m[name].raw_tril.grad).all()
print("03 SPD3 modules: PASS (init cov, PSD, CUDA float64, differentiable)")

03 SPD3 modules: PASS (init cov, PSD, CUDA float64, differentiable)


/tmp/ipykernel_34772/4086449628.py:68: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  assert float((C - INIT_STD[name] ** 2 * torch.eye(3, **DD)).abs().max()) < 1e-2 * INIT_STD[name] ** 2


Regularizers for scale, correlation, condition number, and innovation consistency.


In [4]:
# 04 regularizers: log-eigenvalue prior, correlation, condition number, NIS
from estimation_calibration_cuda.covariance_calibration import (
    LAMBDA, BIAS_PRIOR_BOOST, MAX_LOG_COND,
    reg_correlation, reg_condition_number, reg_nis, covariance_regularization,
)

# unit checks: at init the prior and correlation terms are ~0; a strongly
# correlated matrix is penalized; an ill-conditioned one trips the cond term
_m = make_cov_modules(device=device)
_tot, _terms = covariance_regularization(_m, [], [], device=device)
# near zero at init (small residual from the softplus floor offset)
assert float(_tot) < 1e-4, float(_tot)
_corr_mat = torch.tensor([[1.0, 0.9, 0.0], [0.9, 1.0, 0.0], [0.0, 0.0, 1.0]], **DD)
assert abs(float(reg_correlation(_corr_mat)) - 2 * 0.81 / 9) < 1e-12
_ill = torch.diag(torch.tensor([1.0, 1.0, 1e-4], **DD))
assert float(reg_condition_number(_ill)) > 5.0
assert float(reg_nis([torch.tensor(6.0, **DD)], [3], device=device)) == 1.0
print("04 regularizers: PASS")

04 regularizers: PASS


Seven rollouts, trimmed 1.0 s at each end, with fixed contact schedules. Initial covariances are saved before training.


In [5]:
# 05 data: rollouts, trimming, and contact events
from estimation_calibration_cuda.covariance_calibration import (
    load_rollouts, fixed_initial_covariance, seed_state, eval_replay, save_covariances_npz,
)

TRIM_S, S_JITTER = CONFIG.trim_s, CONFIG.s_jitter
ROLLOUT_ORDER, SPLIT_LABEL, ROLLOUTS = load_rollouts(DATA_ROOT, config=CONFIG, device=device)
assert len(ROLLOUT_ORDER) == 7, ROLLOUT_ORDER

TOTAL_TRAIN_ROWS = sum(r.trim1 - r.trim0 - 1 for r in ROLLOUTS.values())
print(f"{len(ROLLOUT_ORDER)} rollouts, trim {TRIM_S}s "
      f"({ROLLOUTS[ROLLOUT_ORDER[0]].trim0} rows) each end")
for stem in ROLLOUT_ORDER:
    r = ROLLOUTS[stem]
    print(f"  {stem} [{SPLIT_LABEL[stem]}]: rows {r.trim0}..{r.trim1} of {r.total_rows}")
print(f"rows contributing to training loss per epoch: {TOTAL_TRAIN_ROWS} "
      "(the seed row of each rollout is the GT-seeded state itself)")

P0_FIXED = fixed_initial_covariance(device)

# initial (pre-training) covariances: freeze and SAVE before any training
_m0 = make_cov_modules(device=device)
with torch.no_grad():
    covs0, Rk0 = build_covs(_m0)
    covs0 = {k: v.detach().clone() for k, v in covs0.items()}
    Rk0 = Rk0.detach().clone()
save_covariances_npz(OUT / "initial_covariances.npz", covs0, Rk0)
print("initial covariances saved to", OUT / "initial_covariances.npz")

7 rollouts, trim 1.0s (50 rows) each end
  dance1_subject1_20260623_173019 [train]: rows 50..6524 of 6574
  dance1_subject3_20260623_164158 [val]: rows 50..6524 of 6574
  run1_subject2_20260623_170820 [train]: rows 50..11840 of 11890
  run2_subject1_20260623_165739 [test]: rows 50..12190 of 12240
  walk1_subject5_20260623_164533 [train]: rows 50..13015 of 13065
  walk2_subject3_20260623_171914 [train]: rows 50..11859 of 11909
  walk2_subject4_20260623_163125 [test]: rows 50..11859 of 11909
rows contributing to training loss per epoch: 73454 (the seed row of each rollout is the GT-seeded state itself)
initial covariances saved to runs/covariance_calibration/initial_covariances.npz


Runtime profile of one training-shaped chunk.


In [6]:
# 05b performance profile of one training-shaped chunk
_roll = ROLLOUTS[ROLLOUT_ORDER[0]]
_a, _b = _roll.trim0 + 1, _roll.trim0 + 201
_L = torch.linalg.cholesky(covs0["Qc"]).requires_grad_(True)
_covs = dict(covs0)

def _bench(with_grad):
    torch.cuda.synchronize(); t0 = time.time()
    _covs["Qc"] = _L @ _L.T
    X0, th0, P0 = seed_state(_roll, _roll.trim0, P0_FIXED)
    filt = start_filter(X0, th0, P0, _covs, _roll.flags[_roll.trim0],
                        _roll.p_BC[_roll.trim0], Rk0, s_jitter=S_JITTER)
    ctx = torch.enable_grad() if with_grad else torch.no_grad()
    with ctx:
        out = run_rows(filt, _roll.imu[_a:_b], _roll.dt, _roll.p_BC[_a:_b],
                       None, None, Rk0, changes_list=_roll.changes[_a:_b])
        if with_grad:
            (out["v_W"] ** 2).sum().backward()
            _L.grad = None
    torch.cuda.synchronize()
    return (time.time() - t0) / (_b - _a) * 1e3

_bench(True)  # warmup
fwd_bwd_ms = _bench(True)
fwd_ms = _bench(False)
print(f"training-shaped chunk: fwd+bwd {fwd_bwd_ms:.1f} ms/step, "
      f"no-grad {fwd_ms:.1f} ms/step")
print("bottleneck: host-side kernel-launch overhead of the dynamic-dimension "
      "sequential filter; backward dominates. Safe caching optimizations are "
      "already applied in src/estimation_calibration_cuda/invariant_ekf.py (values bitwise-verified in "
      "cell 02); further gains need an estimator-level redesign.")


training-shaped chunk: fwd+bwd 7.9 ms/step, no-grad 2.2 ms/step
bottleneck: host-side kernel-launch overhead of the dynamic-dimension sequential filter; backward dominates. Safe caching optimizations are already applied in src/estimation_calibration_cuda/invariant_ekf.py (values bitwise-verified in cell 02); further gains need an estimator-level refactor (documented in the header cell).


Run chunked-BPTT covariance training and save the selected checkpoint.


In [7]:
# 06 trimmed training: continuous per-rollout replay, chunked BPTT,
# one Adam update per chunk, selection by epoch-mean training body loss.
# This cell launches the full training run.
from estimation_calibration_cuda.covariance_calibration import (
    train_trimmed_rollouts, save_training_outputs,
)

# Use the completed run's learning rate; retry lower only on non-finite loss.
try:
    result = train_trimmed_rollouts(ROLLOUT_ORDER, ROLLOUTS, config=CONFIG, device=device)
except FloatingPointError as e:
    print(f"non-finite loss at lr={CONFIG.lr} ({e}); retrying with lr={CONFIG.fallback_lr}")
    CONFIG = replace(CONFIG, lr=CONFIG.fallback_lr)
    result = train_trimmed_rollouts(ROLLOUT_ORDER, ROLLOUTS, config=CONFIG, device=device)

RUN_META = save_training_outputs(
    OUT,
    config=CONFIG,
    split_labels=SPLIT_LABEL,
    rollout_order=ROLLOUT_ORDER,
    rollouts=ROLLOUTS,
    result=result,
    initial_covs=covs0,
    initial_R_kin=Rk0,
    device_name=torch.cuda.get_device_name(0),
)
cov_modules, history, best = result.modules, result.history, result.best
print(f"selected epoch {best['epoch']} by train body loss {best['loss']:.4f} "
      f"(epoch 0: {history[0]['train_body_loss']:.4f}) | "
      f"runtime {result.runtime_s / 60:.1f} min")
print("checkpoint (selected + final + optimizer state) saved to",
      OUT / "calibration_checkpoint.pt")
assert best["loss"] < history[0]["train_body_loss"], "training loss must decrease"
assert all(np.isfinite([h["train_body_loss"] for h in history]))
for n in GROUP_ORDER:
    gmean = np.mean([h["groups"][n]["grad_norm_mean"] for h in history])
    print(f"mean grad norm [{n}]: {gmean:.3e}")
assert np.mean([h["groups"]["kin_meas"]["grad_norm_mean"] for h in history]) > 0
assert np.mean([h["groups"]["contact_proc"]["grad_norm_mean"] for h in history]) > 0

epoch  0: body 3.7250 reg 0.00183 | NIS/dim 0.126 | 246s (298 rows/s) | peak 0.16 GB


epoch  1: body 3.2996 reg 0.00374 | NIS/dim 0.108 | 246s (298 rows/s) | peak 0.16 GB


epoch  2: body 2.9987 reg 0.00572 | NIS/dim 0.093 | 247s (297 rows/s) | peak 0.16 GB


epoch  3: body 2.8811 reg 0.00723 | NIS/dim 0.085 | 247s (298 rows/s) | peak 0.16 GB


epoch  4: body 2.8450 reg 0.00826 | NIS/dim 0.081 | 247s (297 rows/s) | peak 0.16 GB


...

epoch 15: body 4.9874 reg 0.01462 | NIS/dim 0.058 | 247s (297 rows/s) | peak 0.16 GB


epoch 16: body 5.1060 reg 0.01490 | NIS/dim 0.055 | 246s (298 rows/s) | peak 0.16 GB


epoch 17: body 5.2051 reg 0.01515 | NIS/dim 0.054 | 246s (299 rows/s) | peak 0.16 GB


epoch 18: body 5.2903 reg 0.01538 | NIS/dim 0.052 | 244s (301 rows/s) | peak 0.16 GB


epoch 19: body 5.3648 reg 0.01559 | NIS/dim 0.051 | 245s (300 rows/s) | peak 0.16 GB
selected epoch 5 by train body loss 2.8429 (epoch 0: 3.7250) | runtime 82.1 min
checkpoint (selected + final + optimizer state) saved to runs/covariance_calibration/calibration_checkpoint.pt
mean grad norm [gyro]: 3.221e-02
mean grad norm [accel]: 8.876e-02
mean grad norm [gyro_bias]: 2.153e-05
mean grad norm [accel_bias]: 2.238e-05
mean grad norm [contact_proc]: 1.540e-01
mean grad norm [kin_meas]: 2.018e-01
